---
# Amyloid Core Predictor

Predict **residue-level amyloid fibril-core propensity** from a protein sequence using **ESM-2** and **ANKH** sequence representations with trained ExtraTrees classifiers.

- **Input:** UniProt ID or protein sequence
- **Models:** ESM-2, ANKH, or both
- **Window sizes:** 9, 15, 21 residues, or all three
- **Output:** per-residue `P_CORE` profiles, predicted CORE segments, interactive graph, and downloadable results
- **How to run:** `Runtime` → `Run all` (or `Ctrl/Cmd + F9`)

`P_CORE` is the ExtraTrees CORE-class probability score and should be interpreted as a model score rather than a separately calibrated biochemical probability.

---

In [ ]:
#@title 1. Protein input { display-mode: "form", form-width: "100%" }

#@markdown Enter a UniProt accession **or** paste a protein sequence. If both are supplied, the pasted sequence is used.
uniprot_id = ""  #@param {type:"string"}
input_sequence = ""  #@param {type:"string"}

#@markdown Choose the representation(s).
select_embedding(s) = "Both"  #@param ["Both", "ESM2", "ANKH"]

#@markdown Choose the window size(s).
select_window = "All"  #@param ["All", "W9", "W15", "W21"]

#@markdown CORE-call threshold. Default: 0.50.
core_threshold = 0.50  #@param {type:"number"}

#@markdown Optional name for the output files.
query_name = "query"  #@param {type:"string"}

# 2. Predict

In [ ]:
#@title Set up predictor { display-mode: "form", form-width: "100%" }
#@markdown Run this cell or use **Runtime → Run all**.

import gc
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path
from typing import List

# ------------------------------------------------------------------
# Model-bank source
# ------------------------------------------------------------------
# Optional direct URL for the model-bank ZIP.
# If left blank, the notebook can use a local file or request an upload.
PUBLISHED_MODEL_BANK_URL = "https://huggingface.co/datasets/jasdeep002/AmyloCore-ML-models/resolve/main/Final_model_bank.zip"

# A local Colab file with this name is used automatically if present.
LOCAL_MODEL_BANK_ZIP = "/content/Step14A_final_model_bank.zip"

OUTPUT_DIR = Path("/content/AmyloidCorePredictor_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Batch sizes can be lowered if accelerator memory is limited.
ESM2_BATCH_SIZE = 16
ANKH_BATCH_SIZE = 16

REPRESENTATIONS = {
    "ESM2": {
        "model_name": "facebook/esm2_t33_650M_UR50D",
        "dim": 1280,
        "batch_size": ESM2_BATCH_SIZE,
    },
    "ANKH": {
        "model_name": "ElnaggarLab/ankh-large",
        "dim": 1536,
        "batch_size": ANKH_BATCH_SIZE,
    },
}
WINDOWS_REQUIRED = [9, 15, 21]
B_DATASETS = ["B1", "B2", "B3"]
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ")

# Install required packages if they are not already available.
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "joblib": "joblib",
    "torch": "torch",
    "transformers": "transformers",
    "ankh": "ankh",
    "plotly": "plotly",
    "requests": "requests",
}
missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

import numpy as np
import pandas as pd
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Predictor setup ready.")

In [ ]:
#@title Resolve protein sequence { display-mode: "form", form-width: "100%" }

import requests

def clean_sequence(seq):
    s = re.sub(r"\s+", "", str(seq)).upper().replace("*", "")
    if not s:
        raise ValueError("Protein sequence is empty.")
    bad = sorted(set(s) - VALID_AA)
    if bad:
        raise ValueError(
            "Unsupported residue symbols: %s" % ",".join(bad)
        )
    return s

def fetch_uniprot_sequence(accession):
    accession = str(accession).strip()
    if not accession:
        raise ValueError(
            "Enter a UniProt accession or paste a protein sequence."
        )
    url = "https://rest.uniprot.org/uniprotkb/%s.fasta" % accession
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    lines = [
        x.strip()
        for x in response.text.splitlines()
        if x.strip()
    ]
    if not lines or not lines[0].startswith(">"):
        raise RuntimeError("UniProt did not return a FASTA record.")
    seq = "".join(x for x in lines if not x.startswith(">"))
    return clean_sequence(seq)

if str(input_sequence).strip():
    sequence = clean_sequence(input_sequence)
    sequence_source = "pasted sequence"
elif str(uniprot_id).strip():
    sequence = fetch_uniprot_sequence(uniprot_id)
    sequence_source = "UniProt:%s" % str(uniprot_id).strip()
else:
    raise ValueError(
        "Enter a UniProt accession or paste a protein sequence."
    )

query_name = re.sub(
    r"[^A-Za-z0-9_.-]+",
    "_",
    str(query_name).strip() or str(uniprot_id).strip() or "query",
).strip("_") or "query"

if select_embedding(s) == "Both":
    SELECTED_REPS = ["ESM2", "ANKH"]
else:
    SELECTED_REPS = [select_embedding(s)]

if select_window == "All":
    SELECTED_WINDOWS = [9, 15, 21]
else:
    SELECTED_WINDOWS = [int(select_window.replace("W", ""))]

if not (0.0 <= float(core_threshold) <= 1.0):
    raise ValueError("core_threshold must lie between 0 and 1.")

if len(sequence) < min(SELECTED_WINDOWS):
    raise ValueError(
        "Sequence length (%d) is shorter than the selected window."
        % len(sequence)
    )

print("Input:", sequence_source)
print("Length:", len(sequence), "aa")
print("Representation(s):", ", ".join(SELECTED_REPS))
print("Window(s):", ", ".join("W%d" % w for w in SELECTED_WINDOWS))
print("Device:", DEVICE)

In [ ]:
#@title Prediction engine { display-mode: "form", form-width: "100%" }

import joblib

def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        while True:
            b = fh.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def generate_windows(seq, w):
    if len(seq) < w:
        raise ValueError("Sequence length %d < W%d" % (len(seq), w))
    return pd.DataFrame([
        {
            "Window_index": i,
            "Window_start": i + 1,
            "Window_end": i + w,
            "Window_sequence": seq[i:i+w],
        }
        for i in range(len(seq) - w + 1)
    ])

def final_model_paths(bank_root, rep, w, b):
    d = Path(bank_root) / "models" / rep / ("W%d" % w) / b
    stem = "Step14A_%s_W%d_%s_full_development_ExtraTrees" % (
        rep, w, b
    )
    return {
        "model": d / (stem + ".joblib"),
        "manifest": d / (stem + "_manifest.json"),
    }

def find_bank_root(start):
    start = Path(start).expanduser().resolve()
    if (
        (start / "Step14A_package_manifest.json").exists()
        and (start / "models").is_dir()
    ):
        return start
    hits = []
    for p in [start] + [x for x in start.rglob("*") if x.is_dir()]:
        if (
            (p / "Step14A_package_manifest.json").exists()
            and (p / "models").is_dir()
        ):
            hits.append(p.resolve())
    hits = list(dict.fromkeys(hits))
    if len(hits) != 1:
        raise RuntimeError(
            "Expected exactly one Step14A model-bank root; found %d."
            % len(hits)
        )
    return hits[0]

def resolve_model_bank_zip():
    local = Path(LOCAL_MODEL_BANK_ZIP)
    if local.exists():
        print("Using local final model bank.")
        return local

    if str(PUBLISHED_MODEL_BANK_URL).strip():
        target = Path("/content/Step14A_final_model_bank.zip")
        print("Downloading final model bank...")
        urllib.request.urlretrieve(PUBLISHED_MODEL_BANK_URL, target)
        return target

    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "No hosted model-bank URL is configured. "
            "Provide Step14A_final_model_bank.zip at %s."
            % LOCAL_MODEL_BANK_ZIP
        ) from exc

    print(
        "Upload the model-bank ZIP."
    )
    uploaded = files.upload()
    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zips) != 1:
        raise ValueError(
            "Expected exactly one ZIP model bank; received %s." % zips
        )
    target = Path("/content/Step14A_final_model_bank.zip")
    src = Path(zips[0])
    if src.resolve() != target.resolve():
        shutil.copy2(src, target)
    return target

def safe_extract_zip(zip_path, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    root = out_dir.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target = (out_dir / member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(
                    "Unsafe ZIP member: %s" % member.filename
                )
        zf.extractall(out_dir)
    return find_bank_root(out_dir)

def inspect_package_manifest(bank_root):
    p = Path(bank_root) / "Step14A_package_manifest.json"
    manifest = json.loads(p.read_text(encoding="utf-8"))
    checks = [
        (manifest.get("status") == "PASS", "package status"),
        (
            sorted(manifest.get("representations", []))
            == ["ANKH", "ESM2"],
            "representations",
        ),
        (
            sorted(int(x) for x in manifest.get("windows", []))
            == [9, 15, 21],
            "windows",
        ),
        (
            sorted(manifest.get("b_datasets", []))
            == ["B1", "B2", "B3"],
            "B datasets",
        ),
        (int(manifest.get("model_count", -1)) == 18, "model count"),
        (
            manifest.get("fold_models_used_at_inference") is False,
            "fold-model inference flag",
        ),
    ]
    bad = [label for ok, label in checks if not ok]
    if bad:
        raise RuntimeError(
            "Step14A model-bank QC failed: %s" % ", ".join(bad)
        )
    return manifest, p

def ensure_sklearn_version(required_version):
    current = importlib.metadata.version("scikit-learn")
    if current != required_version:
        print(
            "Installing model-bank scikit-learn version %s (current %s)..."
            % (required_version, current)
        )
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--upgrade", "scikit-learn==%s" % required_version
        ])
        current = importlib.metadata.version("scikit-learn")
    if current != required_version:
        raise RuntimeError(
            "scikit-learn mismatch after installation: required=%s current=%s"
            % (required_version, current)
        )
    return current

def load_three_models(bank_root, rep, w):
    import sklearn
    models = []
    meta = []
    for b in B_DATASETS:
        p = final_model_paths(bank_root, rep, w, b)
        if not p["model"].exists() or not p["manifest"].exists():
            raise FileNotFoundError(
                "Missing final model/manifest for %s W%d %s"
                % (rep, w, b)
            )
        man = json.loads(p["manifest"].read_text(encoding="utf-8"))
        expected = {
            "status": "PASS",
            "representation": rep,
            "window_size": int(w),
            "b_dataset": b,
            "model_name": "ExtraTreesClassifier",
            "n_estimators": 800,
            "class_weight": "balanced",
            "unseen_data_used": False,
            "embedding_dim": int(REPRESENTATIONS[rep]["dim"]),
        }
        for key, value in expected.items():
            if man.get(key) != value:
                raise RuntimeError(
                    "%s W%d %s manifest mismatch for %s: %r vs %r"
                    % (rep, w, b, key, man.get(key), value)
                )
        if str(man.get("sklearn_version", "")) != sklearn.__version__:
            raise RuntimeError(
                "%s W%d %s sklearn mismatch" % (rep, w, b)
            )
        observed_sha = sha256_file(p["model"])
        if observed_sha != str(man.get("model_sha256", "")):
            raise RuntimeError(
                "%s W%d %s SHA256 mismatch" % (rep, w, b)
            )
        model = joblib.load(p["model"])
        if model.__class__.__name__ != "ExtraTreesClassifier":
            raise RuntimeError("Unexpected classifier class.")
        if int(model.n_features_in_) != int(REPRESENTATIONS[rep]["dim"]):
            raise RuntimeError("Classifier feature dimension mismatch.")
        if int(model.n_estimators) != 800:
            raise RuntimeError("Classifier tree count mismatch.")
        if model.class_weight != "balanced":
            raise RuntimeError("Classifier class_weight mismatch.")
        if not np.array_equal(np.asarray(model.classes_), np.array([0, 1])):
            raise RuntimeError("Classifier classes are not [0,1].")
        models.append(model)
        meta.append({
            "Representation": rep,
            "Window": w,
            "B_dataset": b,
            "Model_SHA256": observed_sha,
        })
    if len(models) != 3:
        raise RuntimeError("Expected exactly three B models.")
    return models, meta

def predict_three_models(models, X):
    P = np.vstack([
        np.asarray(m.predict_proba(X)[:, 1], dtype=np.float64)
        for m in models
    ])
    if P.shape != (3, len(X)):
        raise RuntimeError(
            "Prediction matrix shape %s; expected %s"
            % (P.shape, (3, len(X)))
        )
    if not np.isfinite(P).all():
        raise RuntimeError("Non-finite prediction values.")
    if (P < 0).any() or (P > 1).any():
        raise RuntimeError("Prediction outside [0,1].")
    return P

def reconstruct_profiles(window_df, model_window_scores, protein_length):
    n_models, n_windows = model_window_scores.shape
    if n_windows != len(window_df):
        raise RuntimeError("Prediction/window count mismatch.")
    sums = np.zeros((n_models, protein_length), dtype=np.float64)
    counts = np.zeros(protein_length, dtype=np.int32)
    for j, row in enumerate(window_df.itertuples(index=False)):
        s0 = int(row.Window_start) - 1
        e0 = int(row.Window_end)
        sums[:, s0:e0] += model_window_scores[:, j][:, None]
        counts[s0:e0] += 1
    if np.any(counts == 0):
        raise RuntimeError("At least one residue received zero window coverage.")
    return sums / counts[None, :], counts

def contiguous_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return []
    breaks = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends = np.r_[idx[breaks], idx[-1]]
    return [
        (int(s + 1), int(e + 1))
        for s, e in zip(starts, ends)
    ]

def clear_accelerator_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def mean_pool_special_mask(hidden, attention_mask, special_tokens_mask):
    mask = attention_mask.bool() & (~special_tokens_mask.bool())
    if (mask.sum(dim=1) == 0).any():
        raise RuntimeError("Zero residue tokens after special-token masking")
    m = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * m).sum(dim=1) / m.sum(dim=1)

def mean_pool_first_lengths(hidden, lengths: List[int]):
    import torch
    out = []
    for i, L in enumerate(lengths):
        if L < 1 or L > hidden.shape[1]:
            raise RuntimeError("Invalid residue length %d for hidden shape %s" % (L, tuple(hidden.shape)))
        out.append(hidden[i, :L].mean(dim=0))
    return torch.stack(out, dim=0)

def prepare_rep_model(rep: str, device):
    cfg = REPRESENTATIONS[rep]
    if rep == "ANKH":
        try:
            import ankh
        except ImportError as exc:
            raise SystemExit("ANKH embedding requires: pip install ankh") from exc
        if hasattr(ankh, "load_ankh_large"):
            model, tokenizer = ankh.load_ankh_large()
        elif hasattr(ankh, "load_large_model"):
            model, tokenizer = ankh.load_large_model()
        else:
            raise RuntimeError("Installed ankh package has no large-model loader")
        model.eval()
        model.to(device)
        return model, tokenizer

    if rep == "ESM2":
        try:
            from transformers import AutoModel, AutoTokenizer
        except ImportError as exc:
            raise SystemExit("ESM2 embedding requires transformers") from exc
        tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], use_fast=False)
        model = AutoModel.from_pretrained(cfg["model_name"])
        model.eval()
        model.to(device)
        return model, tokenizer

    raise ValueError(rep)

def embed_sequences(rep: str, seqs: List[str], model, tokenizer, device, batch_size: int) -> np.ndarray:
    import torch
    all_out = []
    n = len(seqs)
    for start in range(0, n, batch_size):
        stop = min(start + batch_size, n)
        batch = seqs[start:stop]
        if rep == "ANKH":
            token_input = [list(s) for s in batch]
            enc = tokenizer(
                token_input,
                add_special_tokens=True,
                padding=True,
                is_split_into_words=True,
                return_tensors="pt",
                return_attention_mask=True,
                return_special_tokens_mask=True,
            )
            special = enc.pop("special_tokens_mask", None)
            enc = {k: v.to(device) for k, v in enc.items()}
            if special is not None:
                special = special.to(device)
            with torch.inference_mode():
                output = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
                hidden = output.last_hidden_state
                if special is not None:
                    pooled = mean_pool_special_mask(hidden, enc["attention_mask"], special)
                else:
                    pooled = mean_pool_first_lengths(hidden, [len(s) for s in batch])
        elif rep == "ESM2":
            enc = tokenizer(
                batch,
                padding=True,
                truncation=False,
                return_tensors="pt",
                return_attention_mask=True,
                return_special_tokens_mask=True,
            )
            special = enc.pop("special_tokens_mask")
            enc = {k: v.to(device) for k, v in enc.items()}
            special = special.to(device)
            with torch.inference_mode():
                output = model(**enc)
                pooled = mean_pool_special_mask(output.last_hidden_state, enc["attention_mask"], special)
        else:
            raise ValueError(rep)

        arr = pooled.detach().float().cpu().numpy().astype(np.float32, copy=False)
        if arr.ndim != 2 or arr.shape[1] != int(REPRESENTATIONS[rep]["dim"]):
            raise RuntimeError("%s: unexpected embedding shape %s" % (rep, arr.shape))
        if not np.isfinite(arr).all():
            raise RuntimeError("%s: non-finite embeddings" % rep)
        all_out.append(arr)
        print("      %s: embedded %d/%d unique sequences" % (rep, stop, n), flush=True)
    if not all_out:
        raise RuntimeError("%s: no sequences to embed" % rep)
    return np.concatenate(all_out, axis=0).astype(np.float32, copy=False)

print("Prediction engine ready.")

In [ ]:
#@title Load model bank { display-mode: "form", form-width: "100%" }

bank_zip = resolve_model_bank_zip()
bank_root = safe_extract_zip(
    bank_zip,
    OUTPUT_DIR / "_model_bank",
)
package_manifest, package_manifest_path = inspect_package_manifest(bank_root)
runtime_sklearn = ensure_sklearn_version(
    str(package_manifest["sklearn_version"])
)

print("Model bank loaded.")
print("scikit-learn:", runtime_sklearn)

In [ ]:
#@title Run prediction { display-mode: "form", form-width: "100%" }

all_residue = []
all_segments = []
model_manifest_rows = []

print("\nRunning amyloid-core predictor...")
print("Sequence length:", len(sequence))

for rep in SELECTED_REPS:
    print("\nLoading %s..." % rep, flush=True)
    plm_model, tokenizer = prepare_rep_model(rep, DEVICE)
    cfg = REPRESENTATIONS[rep]

    for w in SELECTED_WINDOWS:
        if len(sequence) < w:
            print("%s W%d skipped: sequence shorter than window." % (rep, w))
            continue

        print("%s W%d: generating stride-1 windows..." % (rep, w), flush=True)
        wdf = generate_windows(sequence, w)

        print(
            "%s W%d: generating fresh embeddings (%d windows)..."
            % (rep, w, len(wdf)),
            flush=True,
        )
        X = embed_sequences(
            rep,
            wdf["Window_sequence"].tolist(),
            plm_model,
            tokenizer,
            DEVICE,
            int(cfg["batch_size"]),
        )

        expected_shape = (len(wdf), int(cfg["dim"]))
        if tuple(X.shape) != expected_shape:
            raise RuntimeError(
                "%s W%d embedding shape %s; expected %s"
                % (rep, w, X.shape, expected_shape)
            )

        models, meta = load_three_models(bank_root, rep, w)
        model_manifest_rows.extend(meta)

        P = predict_three_models(models, X)
        profiles, coverage = reconstruct_profiles(
            wdf,
            P,
            len(sequence),
        )

        p_core = profiles.mean(axis=0)
        b_sd = profiles.std(axis=0, ddof=0)
        support = (profiles >= float(core_threshold)).mean(axis=0)
        pred = p_core >= float(core_threshold)

        rdf = pd.DataFrame({
            "Position": np.arange(1, len(sequence) + 1, dtype=int),
            "AA": list(sequence),
            "Representation": rep,
            "Window": "W%d" % w,
            "P_CORE": p_core,
            "B_Model_SD": b_sd,
            "B_support_fraction_at_threshold": support,
            "Predicted_CORE": pred.astype(int),
            "Window_coverage": coverage,
        })
        all_residue.append(rdf)

        for seg_i, (s, e) in enumerate(contiguous_segments(pred), start=1):
            all_segments.append({
                "Representation": rep,
                "Window": "W%d" % w,
                "Segment_ID": seg_i,
                "Start": s,
                "End": e,
                "Length": e - s + 1,
                "Mean_P_CORE": float(p_core[s-1:e].mean()),
                "Max_P_CORE": float(p_core[s-1:e].max()),
            })

        del X, P, profiles, models
        clear_accelerator_cache()

    del plm_model, tokenizer
    clear_accelerator_cache()

if not all_residue:
    raise RuntimeError("No prediction profile was generated.")

residue_long = pd.concat(all_residue, ignore_index=True)
segments_df = pd.DataFrame(
    all_segments,
    columns=[
        "Representation", "Window", "Segment_ID",
        "Start", "End", "Length",
        "Mean_P_CORE", "Max_P_CORE"
    ],
)

# Build one compact wide table.
wide = pd.DataFrame({
    "Position": np.arange(1, len(sequence) + 1, dtype=int),
    "AA": list(sequence),
})
for rep in SELECTED_REPS:
    for w in SELECTED_WINDOWS:
        label = "W%d" % w
        sub = residue_long[
            (residue_long["Representation"] == rep)
            & (residue_long["Window"] == label)
        ][["Position", "P_CORE", "B_Model_SD", "Predicted_CORE"]].copy()
        if sub.empty:
            continue
        sub.rename(
            columns={
                "P_CORE": "%s_%s_P_CORE" % (rep, label),
                "B_Model_SD": "%s_%s_B_Model_SD" % (rep, label),
                "Predicted_CORE": "%s_%s_Predicted_CORE" % (rep, label),
            },
            inplace=True,
        )
        wide = wide.merge(
            sub,
            on="Position",
            how="left",
            validate="one_to_one",
        )

print("\nPrediction complete.")

# 3. Results

In [ ]:
#@title Visualize residue-level CORE probability { display-mode: "form", form-width: "100%" }

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

reps_present = [
    rep for rep in ["ESM2", "ANKH"]
    if rep in set(residue_long["Representation"])
]

fig = make_subplots(
    rows=len(reps_present),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10 if len(reps_present) > 1 else 0.04,
    subplot_titles=[
        "%s residue-level CORE probability" % rep
        for rep in reps_present
    ],
)

for row_i, rep in enumerate(reps_present, start=1):
    for w in SELECTED_WINDOWS:
        label = "W%d" % w
        sub = residue_long[
            (residue_long["Representation"] == rep)
            & (residue_long["Window"] == label)
        ].sort_values("Position")
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["Position"],
                y=sub["P_CORE"],
                mode="lines",
                name="%s %s" % (rep, label),
                legendgroup="%s_%s" % (rep, label),
                hovertemplate=(
                    "Residue %%{x}<br>"
                    "P_CORE %%{y:.3f}<extra>%s %s</extra>"
                    % (rep, label)
                ),
            ),
            row=row_i,
            col=1,
        )

    fig.add_hline(
        y=float(core_threshold),
        line_dash="dash",
        annotation_text="threshold %.2f" % float(core_threshold),
        annotation_position="top right",
        row=row_i,
        col=1,
    )
    fig.update_yaxes(
        title_text="P_CORE",
        range=[0, 1],
        row=row_i,
        col=1,
    )

fig.update_xaxes(
    title_text="Residue position",
    row=len(reps_present),
    col=1,
)

fig.update_layout(
    title=(
        "Amyloid Core Predictor — %s (%d aa)"
        % (query_name, len(sequence))
    ),
    template="plotly_white",
    height=430 if len(reps_present) == 1 else 760,
    hovermode="x unified",
    legend_title_text="Model / window",
)

fig.show()

print("\nPredicted CORE segments at threshold %.2f" % float(core_threshold))
if len(segments_df):
    display(
        segments_df.sort_values(
            ["Representation", "Window", "Start"]
        ).reset_index(drop=True)
    )
else:
    print("No contiguous segment crossed the selected threshold.")

print("\nPer-residue results preview")
display(wide.head(25))

In [ ]:
#@title Download results { display-mode: "form", form-width: "100%" }

import datetime

stem = re.sub(
    r"[^A-Za-z0-9_.-]+",
    "_",
    query_name,
).strip("_") or "query"

residue_csv = OUTPUT_DIR / ("%s_residue_predictions.csv" % stem)
segments_csv = OUTPUT_DIR / ("%s_predicted_segments.csv" % stem)
plot_html = OUTPUT_DIR / ("%s_probability_profiles.html" % stem)
manifest_json = OUTPUT_DIR / ("%s_run_manifest.json" % stem)
bundle_zip = OUTPUT_DIR / ("%s_AmyloidCorePredictor_results.zip" % stem)

wide.to_csv(residue_csv, index=False)
segments_df.to_csv(segments_csv, index=False)
fig.write_html(str(plot_html), include_plotlyjs="cdn")

manifest = {
    "status": "PASS",
    "query_name": stem,
    "sequence_source": sequence_source,
    "sequence_length": int(len(sequence)),
    "sequence_sha256": hashlib.sha256(
        sequence.encode("utf-8")
    ).hexdigest(),
    "representations": SELECTED_REPS,
    "windows": SELECTED_WINDOWS,
    "threshold": float(core_threshold),
    "deployment_rule":
        "arithmetic mean of B1/B2/B3 ExtraTrees probabilities",
    "residue_reconstruction":
        "mean of all stride-1 windows covering each residue",
    "ESM2_model": REPRESENTATIONS["ESM2"]["model_name"],
    "ANKH_model": REPRESENTATIONS["ANKH"]["model_name"],
    "model_bank_sha256": sha256_file(bank_zip),
    "model_bank_manifest_sha256": sha256_file(package_manifest_path),
    "scikit_learn_version": runtime_sklearn,
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
}
manifest_json.write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

if bundle_zip.exists():
    bundle_zip.unlink()

with zipfile.ZipFile(
    bundle_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for path in [
        residue_csv,
        segments_csv,
        plot_html,
        manifest_json,
    ]:
        zf.write(path, arcname=path.name)

print("Results saved:")
print(" ", residue_csv.name)
print(" ", segments_csv.name)
print(" ", plot_html.name)
print(" ", manifest_json.name)
print(" ", bundle_zip.name)
print("ZIP SHA256:", sha256_file(bundle_zip))

try:
    from google.colab import files
    files.download(str(bundle_zip))
except ImportError:
    pass

---
### Output interpretation

- `P_CORE` is the residue-level score reconstructed from all stride-1 windows covering that residue.
- For each representation/window, the deployed score is the arithmetic mean of three B1/B2/B3 ExtraTrees classifiers.
- The default threshold is **0.50**.
- ESM-2 and ANKH are independent representations; selecting **Both** reports both rather than averaging the two PLMs together.
---